# Pneumonia-X — étape 5 : démo en ligne (Hugging Face Spaces) avec Grad-CAM

Avant de lancer :
1. **Hugging Face** (gratuit) : *Settings → Access Tokens → Create new token*, type **Write**. Copier le jeton.
2. **Kaggle** : menu *Add-ons → Secrets → Add secret*, nom `HF_TOKEN`, valeur = le jeton ; cocher la case pour l'attacher à ce notebook.
3. *Session options* : **Internet On** (pas besoin de GPU).
4. **Add Input** : *Your Work → Notebooks* → le notebook de l'étape 4 (sa **Version 1**, qui contient `checkpoints/`), **et** `Chest X-Ray Images (Pneumonia)`.
5. *Run All* (environ 5 min).

In [ ]:
import glob
print(glob.glob('/kaggle/input/**/densenet121.ckpt', recursive=True))

In [ ]:
%cd /kaggle/working
!rm -rf pneumonia && git clone -q https://github.com/khadijafair/pneumonia.git
%cd /kaggle/working/pneumonia
!git log --oneline -1
%env PYTHONPATH=/kaggle/working/pneumonia/src
!pip install -q -r requirements-train.txt gradio huggingface_hub

## 1. Exporter le DenseNet121 (poids seuls, format safetensors) et son seuil

In [ ]:
CKPT = glob.glob('/kaggle/input/**/densenet121.ckpt', recursive=True)[0]
!python scripts/export_model.py --ckpt {CKPT} --pred-val results_rsna/densenet121/pred_val.csv --external results_rsna/evaluation_external.json --out export/densenet121

## 2. Trois exemples du test interne (un normal, une pneumonie bactérienne, une virale)

In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image
from pneumonia.data.index import find_dataset_root

root = find_dataset_root(Path('/kaggle/input'))
pred = pd.read_csv('results_rsna/densenet121/pred_test.csv')
picks = {
    'normal': pred[pred.label == 'NORMAL'].nsmallest(1, 'prob'),
    'pneumonie_bacterienne': pred[pred.subtype == 'bacteria'].nlargest(1, 'prob'),
    'pneumonie_virale': pred[pred.subtype == 'virus'].nlargest(1, 'prob'),
}
ex = Path('demo/examples'); ex.mkdir(exist_ok=True)
for name, row in picks.items():
    img = Image.open(root / row.path.iloc[0]).convert('L')
    img.thumbnail((768, 768))
    img.save(ex / f'{name}.jpeg', quality=90)
    print(name, row.path.iloc[0], round(float(row.prob.iloc[0]), 4))

## 3. Vérification locale avant publication

In [ ]:
import os, sys
os.environ['MODEL_DIR'] = 'export/densenet121'
sys.path.insert(0, 'demo')
import app
for f in sorted(Path('demo/examples').glob('*.jpeg')):
    probs, heat, text = app.predict(Image.open(f))
    print(f.name, round(probs['pneumonie'], 4))
Image.fromarray(heat)

## 4. Publication sur Hugging Face (modèle + Space)

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
!python scripts/publish_hf.py --model-dir export/densenet121 --examples-dir demo/examples